In [1]:
# # Core imports man
# import sys
# import os
# import random
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import scipy
# import time

# # Numerical libraries
# from scipy.integrate import solve_ivp, odeint, cumulative_trapezoid
# from scipy.interpolate import (
#     UnivariateSpline, splrep, splev, CubicSpline, 
#     interp1d, PchipInterpolator, InterpolatedUnivariateSpline
# )

# import numdifftools as nd

# # Random number generator (GSL)
# import pygsl.rng

# # custom InflationModels code to path the one below is for wkb approximation method
# sys.path.append(
#     '/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/InflationModels'
# )

# #the path below assumes the original numerics from Deyan's code
# # sys.path.append('/Users/epmeador/Desktop/InflationModels-master')

# # Enable spectra
# SPECTRUM = True

# # Local modules from InflationModels
# from MacroDefinitions import *
# from calcpath import *
# from int_de import *
# if SPECTRUM:
# #     from spectrum_noS0_fixedNstar import * #this is mode modified one that I have been working with for wkb approx
# #     from spectrum_OG import * #this is the original spectrum from full numerical code
# #     from spectrum_pyoscode_tensor_clean import * #this is the spectrum from pyoscode 
# #     from spectrum_pyoscode_scalar import * #this is the spectrum from pyoscode with scalar 
# #     from spectrum_pyoscode_optimized_wkb import *
# #     from spectrum_pyoscode_test_N import *
#     from spectrum_OG_nanoscale_nodiagnostics import * #this is equivalent to the original spectrum from full numerical code w/o diagnostics

# # ========================
# # GLOBAL SETTINGS
# # ========================
# NEQS = 7  # Updated from 6 to 7
# SPECTRUM = True
# SAVEPATHS = True

# NMAX = 1.2
# NMIN = 0.3

# #if I want to expand the lam4 space for higgs values I can do a symmetric expansion around what I know works
# LAM4_MIN = -1.0e-7   # ~1.5x more negative than base
# LAM4_MAX = 2.0e-7   # 3x current upper bound
# LAM4_BASE = 6.87065e-08
# NUM_LAM4_GRID = 398  # Updated from NUM_LAM3_GRID to NUM_LAM4_GRID
# lam4_set = np.linspace(LAM4_MIN, LAM4_MAX, num=NUM_LAM4_GRID)
# lam4_set = np.sort(np.append(lam4_set, [LAM4_BASE, 0.0]))

# print(f"Total models: {len(lam4_set)}")  # Should be 400 (or 399-400)
# print(f"Base λ₄={LAM4_BASE:.6e} included: {LAM4_BASE in lam4_set}")
# print(f"Zero included: {0.0 in lam4_set}")

# #This sets how many nontrivial models you are running (interesting models you want)
# NUMPOINTS = 1        # number of nontrivial models per λ4

# #This will give us the min and max number of e-folds we are looking for
# NUMEFOLDSMAX = 65.0
# NUMEFOLDSMIN = 57.0

# # #Here we are writing out our output files we can name to analyze data
# BASE_OUTDIR = "/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs7"  # Updated from neqs6 to neqs7

# # RNG initialization process
# #This can be used to generate a random generator that works well with simulations
# my_random = pygsl.rng.ranlxd2()
# my_random.set(0)
# np.random.seed(0)

# # ========================
# # Support Functions
# # ========================

# #Here we define a class for several variables, this will initialize several variables
# #The size we are setting for Y and initY is the same as NEQs which may be number of equations and is set in flow.py
# #We set a state, the initial state, an empty string in the class, number of points, and e-folds
# class Calc:
#     def __init__(self):
#         self.Y = np.zeros(NEQS, dtype=float, order='C')
#         self.initY = np.zeros(NEQS, dtype=float, order='C')
#         self.ret = ""
#         self.npoints = 0
#         self.Nefolds = 0.0

# #Below we are initializing our starting slow roll parameter values
# #We should be able to choose what ell we are extending to.
# def pick_init_vals(lam4):  # Updated from lam3 to lam4

#     #This is what is making it slow I think, uncomment init_vals below
#    # current higgs values
#     init_vals = np.zeros(NEQS, dtype=float, order='C')
#     init_vals[0] = 5.5 #phi0 #i dont think this really affects it??
#     init_vals[1] = 1.0 #H0
#     init_vals[2] = 0.000209237 #epsilon0
#     init_vals[3] = -0.0342419 #sigma0
#     init_vals[4] = 0.000278972#2lam0
#     init_vals[5] = -4.60971e-06  # 3lam
#     init_vals[6] = lam4  # Updated from lam3 to lam4

    
# #     #Below is the original set of values used in your code
# #         #original values 
# #     init_vals = np.zeros(NEQS, dtype=float, order='C')
# #     init_vals[0] = 0.0
# #     init_vals[1] = 1.0
# #     init_vals[2] = my_random.uniform() * 0.8
# #     init_vals[3] = my_random.uniform() - 0.5
# #     init_vals[4] = my_random.uniform()*0.1 - 0.05

# #     prefact = 0.05
    
# #     for i in range (5 , NEQS):
# #         init_vals[i] = my_random.uniform() * prefact - (0.5*prefact)
# #         prefact *= 0.1
       
#     init_Nefolds = my_random.uniform() * (NUMEFOLDSMAX - NUMEFOLDSMIN) + NUMEFOLDSMIN
#     return init_vals, init_Nefolds

# #SAVING PATHS
# #This next part of the code will print either asymptote or nontrivial 
# #And decides when to save the code 
# #This is based on what the model itself is doing

# #This function below will calculation the spectrum computed from y
# #As long as its in a particular range
# #This is where I would limit my range of spectrum models of interest
# #Otherwise it returns false
# def we_should_calc_spec(y):
#     return (specindex(y) > NMIN and specindex(y) < NMAX)

# #Specifically, we will save a model with some interesting dynamics
# #And this means either asymptote or nontrivial - for us nontrivial 
# #Also only if the path has not been saved yet, and only for 
# #Every nth successful model
# #This governs path saving in the non-spectral case
# def we_should_save_path(retval, save, pointcount, printevery):
#     return (retval == "nontrivial") and (not save) and (pointcount % printevery == 0)

# #Overall, this writes model trajectory to a file
# #the SRP at each integration step, the number of N remaining, gives a reconstructed V, and e_H
# # Open output file
# def save_path(y, N, kount, fname):
#     with open(fname, "w") as outfile:
#     # Output intermediate data from the integration
#         for i in range(kount):
#             for j in range(NEQS):
#             #get y variable as a function of i in kount
#             #should be like SRP as a function of time
#             #j loops over NEQS which are used 
#                 outfile.write("%le " % y[j, i])
#             outfile.write("%lf " % N[i])
#             #Will also write out N at that time step
# #             V = 3 * y[1, i]**2 * (1. - y[2, i]/3.) #this one is wrong, i think this is what he had
#             V = (3./(8.*np.pi)) * y[1, i] * y[1, i] * (1.-y[2, i]/3.) #what the original code had
#             outfile.write("%le %le\n" % (V, (V*y[2, i])/(3. - y[2, i]))) #I think this is KE

# # ========================
# # Main Loop
# # ========================
# def run_neqs7_models():  # Updated from run_neqs6_models to run_neqs7_models
#     summary_records = []
    
#     for lam4 in lam4_set:  # Updated from lam3 to lam4
#         print(f"\n=== Running λ4 = {lam4:.1e} ===")

#         # Output directories
#         OUTDIR = f"{BASE_OUTDIR}/lam4_{lam4:.1e}" # Updated from lam3 to lam4
#         os.makedirs(OUTDIR, exist_ok=True)
#         OUTFILE1_NAME = f"{OUTDIR}/test_nr_neqs{NEQS}.dat"
#         OUTFILE2_NAME = f"{OUTDIR}/test_esigma_neqs{NEQS}.dat"

#         # Open output files for writing
#         try:
#             outfile1 = open(OUTFILE1_NAME, "w")
#             outfile2 = open(OUTFILE2_NAME, "w")
#         except IOError as e:
#             print("Could not open output files: ", e)
#             sys.exit()

#         # Spectrum arrays
#         if SPECTRUM:
#             #Scalar mode function
#             #u_s = np.empty((2, knos))
#             u_s = np.empty((2, knos))
#             #Tensor mode function
#             #u_t = np.empty((2, knos))
#             u_t = np.empty((2, knos))
#             #This is an empty array of size NEQS + 1
#             #This can be used to store slow roll variables plus some extra variable, maybe N 
#             y_final = np.empty(NEQS + 1)
#             #if i want to save raw power spectra as well
# #             u_s_raw = np.empty((2, knos))
# #             u_t_raw = np.empty((2, knos))
#             #Below will track how many spectra have been saved
#             spec_count = 0

#         # Initialize counters and allocate buffers(?)
#         #sets everything to 0
#         # iters = total number of iterations
#         calc = Calc()
#         iters = 0
#         points = 0
#         outcount = 0
#         asymcount = 0
#         nontrivcount = 0
#         insuffcount = 0
#         noconvcount = 0
#         badncount = 0
#         errcount = 0
#         savedone = 0

#         # Trial loop
#         #Main part of MC loop: tries random models and decides whether to keep or discard them
#         #This will keep looping as long as nontrivcount is less than NUMPOINTS
#         #So it will find random inflation models until I have found enough nontrivial ones
#         #nontrivial is an interesting model, if valid it will increment nontrivcount until you get NUMPOINTS
#         #loop will stop once you get NUMPOINTS OR hit iters>200
#         while nontrivcount < NUMPOINTS:
#             iters += 1
#             if iters > 200:  # failsafe
#                 break

#             #this will print progress updates
#             #every 100 iterations do something, every 1000 print status report
#             #if multiple of 100 not 1000 print .
#             if iters % 100 == 0:
#                 print(f"  Iter {iters}, nontriv={nontrivcount}")

#             # pick initial conditions right now it is not random
#             yinit, calc.Nefolds = pick_init_vals(lam4)  # Updated from lam3 to lam4
#             #creates copy of random initial conditions created by pick_init_vals
#             y = yinit.copy()

#             # state arrays
#             path = np.array([[]])
#             N = np.array([])

#             # integrate flow equations
#             #this will call the calcpath function using random fluc copy, 
#             #path from srp evolution, N, and calc 
#             # calc stores state! So a string that categorizes this
#             t0 = time.perf_counter()
#             calc.ret = calcpath(calc.Nefolds, y, path, N, calc)
#             t1 = time.perf_counter()
#             print(f"calcpath runtime: {t1 - t0:.4f} s")
#             print(f"  -> {calc.ret}")
            
#             #Confirm diagnostic stuff
#             print("\n=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===")

#             # print initial slow-roll params
#             print(f"Initial values (yinit): {yinit}")
#             print(f"  φ0 = {yinit[0]:.6e}")
#             print(f"  H0 = {yinit[1]:.6e}")
#             print(f"  ε0 = {yinit[2]:.6e}")
#             print(f"  σ0 = {yinit[3]:.6e}")
#             print(f"  λ₂ = {yinit[4]:.6e}")
#             if len(yinit) > 5:
#                 print(f"  λ₄ = {yinit[5]:.6e}")  # Updated from λ₃ to λ₄
#             print(f"Initial Nefolds = {calc.Nefolds:.3f}")

#             # check integrator tolerance (read from pygsl object if available)
#             try:
#                 import inspect
#                 import pygsl.odeiv as odeiv
#                 s = odeiv.step_rk4(len(yinit), derivs1)
#                 c = odeiv.control_y_new(s, 1e-8, 1e-8)  # test new control to show expected
#                 print(f"GSL integrator class: {s.__class__.__name__}")
#                 print(f"Expected tolerances: atol=1e-8, rtol=1e-8")
#             except Exception as e:
#                 print("Could not check GSL integrator:", e)

#             print("===============================================\n")

#             # handle outcome - i told it to get rid of asymptote
#             #the original code kept them for late time attractors
#             #if it is an asymptote we  proceed to find spectral indices and write them to a file
#             #this is done after checking to see if specindex is between NMIN and NMAX
#             #this is our nr file
#             #this asymptote approached a late time attracror, could give valid observables, have to manually screen
#             #mathematically consistent, exact solutions to flow,  usefulf or low-scale inflation models, can correspond to eternal inflation
#             #they dont really allow for reheating which is why i dont like them
#             #removed asymptote
#             if calc.ret == "asymptote":
#                 asymcount += 1
#                 if asymcount > 100:
#                     print("Too many asymptotes, stopping")
#                     break
#                 continue

#             #This block runs if nontrivial, which means it ended inflation after calc.Nefolds, not just asymptote
#             #that block continues below
#             if calc.ret == "nontrivial":
#                 # write observables
#                 r = tsratio(y)
#                 ns = specindex(y)
#                 alpha_s = dspecindex(y)
#                 outfile1.write(f"{r:.10f} {ns:.10f} {alpha_s:.10f}\n")
#                 outfile1.flush()

#                 # write final srp parameters and appends number
#                 for i in range(NEQS):
#                     outfile2.write("%le " % y[i])
#                 outfile2.write("%f\n" % calc.Nefolds)
#                 outfile2.flush()

#                 points += 1 #total valid models saved so far
#                 savedone = 0 #marks that havent saved in teh trajectory files yet
#                 nontrivcount += 1 #counts how many nontrivial models we've accepted

#                 # calculate spectra if needed
#                 if SPECTRUM and we_should_calc_spec(y):
#                     print(f"  ns = {specindex(y):.3f}")
#                     #this tells us what spectrum we are evaluating
#                     print(f"  -> Evaluating spectrum {spec_count}")
#                     #we will need to initalize ps evolution using slow roll values
#                     #time step 3 so that its early enough in inflation so most modes are w/n horizon
#                     y_final[:NEQS] = path[:NEQS, 3] #this grabs all slow roll parameters at time step 3
#                     y_final[NEQS] = N[3] #this add the value of N at that time step
#                     #dont forget : means slicing, path[:NEQS, 3] = path[0:NEQS-1,3]
#                     #which is rows 0-NEQS-1, at column 3
                    
#                     t0 = time.perf_counter()
#                     spectrum_status = spectrum(y_final, y, u_s, u_t, calc.Nefolds,
#                                                derivs1, scalarsys, tensorsys)
# # #if i wanna save the raw power spectrum activate the line below
# #                     spectrum_status = spectrum(y_final, y, u_s, u_t, calc.Nefolds,
# #                                                derivs1, scalarsys, tensorsys,u_s_raw, u_t_raw)
                    
#                     t1 = time.perf_counter()
#                     print(f"spectrum runtime: {t1 - t0:.4f} s")

# #                     #runs through spectrum module which uses that prepared final state
# #                     spectrum_status = spectrum(y_final, y, u_s, u_t, calc.Nefolds,
# #                                                derivs1, scalarsys, tensorsys)
#                     if spectrum_status:
#                         errcount += 1
#                     #if something is wrong it will start counting errors

#                     # save spectra
#                     spec_s_name = f"{OUTDIR}/spec_s{spec_count:03d}_neqs{NEQS}.dat"
#                     spec_t_name = f"{OUTDIR}/spec_t{spec_count:03d}_neqs{NEQS}.dat"
#                     np.savetxt(spec_s_name, u_s[:, :knos].T)
#                     np.savetxt(spec_t_name, u_t[:, :knos].T)
                    
# #                     #if i wanna save raw spectra
# #                     spec_s_raw_name = f"{OUTDIR}/spec_s_raw{spec_count:03d}_neqs{NEQS}.dat"
# #                     spec_t_raw_name = f"{OUTDIR}/spec_t_raw{spec_count:03d}_neqs{NEQS}.dat"

# #                     np.savetxt(spec_s_raw_name, u_s_raw[:, :knos].T)
# #                     np.savetxt(spec_t_raw_name, u_t_raw[:, :knos].T)
#                     spec_count += 1

#                     #Remember we are only computing power spectra for nontrivial models.
#                     #For these models, inflation ends.
#                     #To compute power spectrum you evolve mode k through horizon crossing k=aH
#                     #And you evaluate perturbation amplitudes after horizon exit
#                     #Attractor models never exit inflation

#                 #added a line in calcpath
#                 #normalize path if spectrum=True
#                 if SPECTRUM:
#                     for j in range(calc.npoints):
# # #                         #print("Final Hubble y[1] =", y[1])
# # #                         #this normalization shifts all phi vals
# #                         #it subtracts the final phi from each one so last pt is 0
# #                         #phi is recentered so end of inflation is at phi=0
# #                         path[0, j] -= path[0, calc.npoints-1]
# #                         #we then rescale the hubble values to get correct scale
# #                         #if y[1] is less than zero this makes all H negative 
# # normalize path if spectrum=True (match original)
#                         path[0, :calc.npoints] -= path[0, calc.npoints-1]
#                         path[1, :calc.npoints] *= y[1]
# #                         path[1, j] = path[1, j] * abs(y[1]) #original line 
# #                         path[1, j] *= abs(y[1])
                        
# #                 #Added this        
# #                 Hnorm = 0.00001 * 2 * np.pi * np.sqrt(y[2]) / y[1]
# #                 path[1, :] *= Hnorm

#                 # save path
#                 path_name = f"{OUTDIR}/path_neqs{NEQS}_lam4{lam4:.1e}_{outcount:03d}.dat"  # Updated from lam3 to lam4
#                 print(f"  -> Saving path {path_name}")
#                 save_path(path, N, calc.npoints, path_name)
#                 outcount += 1

#                 # add summary record?
#                 summary_records.append({
#                     "lam4": lam4, "r": r, "n_s": ns,
#                     "alpha_s": alpha_s, "Nefolds": calc.Nefolds
#                 })

#             elif calc.ret == "insuff":
#                 insuffcount += 1
#             elif calc.ret == "noconverge":
#                 noconvcount += 1
#             else:
#                 errcount += 1

#         # close output files
#         outfile1.close()
#         outfile2.close()

#     # write summary CSV
#     summary_df = pd.DataFrame(summary_records)
#     summary_file = f"{BASE_OUTDIR}/neqs{NEQS}_summary.csv"
#     summary_df.to_csv(summary_file, index=False)
#     print(f"\nSummary written to {summary_file}")

# # if __name__ == "__main__":
# #     run_neqs7_models()  # Updated from run_neqs6_models to run_neqs7_models

# print('env', sys.executable)

# import platform, numpy, scipy, pandas, matplotlib
# print(f"Python: {platform.python_version()}")
# print(f"CPU Architecture: {platform.machine()}")
# print(f"NumPy: {numpy.__version__}")
# print(f"SciPy: {scipy.__version__}")
# print(f"pandas: {pandas.__version__}")
# print(f"matplotlib: {matplotlib.__version__}")

# %time run_neqs7_models()  # Updated from run_neqs6_models to run_neqs7_models


In [2]:
# Core imports man
import sys
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import time
import pyoscode

# Numerical libraries
from scipy.integrate import solve_ivp, odeint, cumulative_trapezoid
from scipy.interpolate import (
    UnivariateSpline, splrep, splev, CubicSpline,
    interp1d, PchipInterpolator, InterpolatedUnivariateSpline
)

import numdifftools as nd

# Random number generator (GSL)
import pygsl.rng

# custom InflationModels code to path the one below is for wkb approximation method
sys.path.append(
    '/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/InflationModels'
)

# Enable spectra
SPECTRUM = True

# Local modules from InflationModels
from MacroDefinitions import *
from calcpath import *
from int_de import *
if SPECTRUM:
    from spectrum_OG_nanoscale_nodiagnostics import *  # this is equivalent to the original spectrum from full numerical code w/o diagnostics


# ========================
# GLOBAL SETTINGS
# ========================
NEQS = 7
SPECTRUM = True
SAVEPATHS = True

NMAX = 1.2
NMIN = 0.3

# if I want to expand the lam4 space for higgs values I can do a symmetric expansion around what I know works
LAM4_MIN = -1.0e-7
LAM4_MAX = 2.0e-7
LAM4_BASE = 6.87065e-08
NUM_LAM4_GRID = 248

lam4_set = np.linspace(LAM4_MIN, LAM4_MAX, num=NUM_LAM4_GRID)
lam4_set = np.sort(np.append(lam4_set, [LAM4_BASE, 0.0]))

print(f"Total models: {len(lam4_set)}")
print(f"Base λ₄={LAM4_BASE:.6e} included: {LAM4_BASE in lam4_set}")
print(f"Zero included: {0.0 in lam4_set}")

# This sets how many nontrivial models you are running
NUMPOINTS = 1

# This will give us the min and max number of e-folds we are looking for
NUMEFOLDSMAX = 65.0
NUMEFOLDSMIN = 57.0

# Output directory
BASE_OUTDIR = "/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs7"

# RNG initialization process
my_random = pygsl.rng.ranlxd2()
my_random.set(0)
np.random.seed(0)


# ========================
# Support Functions
# ========================
class Calc:
    def __init__(self):
        self.Y = np.zeros(NEQS, dtype=float, order='C')
        self.initY = np.zeros(NEQS, dtype=float, order='C')
        self.ret = ""
        self.npoints = 0
        self.Nefolds = 0.0


def pick_init_vals(lam4):
    # current Higgs-like values
    init_vals = np.zeros(NEQS, dtype=float, order='C')
    init_vals[0] = 5.5 / np.sqrt(8 * np.pi)   # phi0
    init_vals[1] = 1.0                        # H0
    init_vals[2] = 0.000209237                # epsilon0
    init_vals[3] = -0.0342419                 # sigma0
    init_vals[4] = 0.000278972                # lambda2
    init_vals[5] = -4.60971e-06  # 3lam
    init_vals[6] = lam4  # Updated from lam3 to lam4


    init_Nefolds = my_random.uniform() * (NUMEFOLDSMAX - NUMEFOLDSMIN) + NUMEFOLDSMIN
    return init_vals, init_Nefolds


def we_should_calc_spec(y):
    return (specindex(y) > NMIN and specindex(y) < NMAX)


def we_should_save_path(retval, save, pointcount, printevery):
    return (retval == "nontrivial") and (not save) and (pointcount % printevery == 0)


def save_path(y, N, kount, fname):
    with open(fname, "w") as outfile:
        for i in range(kount):
            for j in range(NEQS):
                outfile.write("%le " % y[j, i])
            outfile.write("%lf " % N[i])

            V = (3. / (8. * np.pi)) * y[1, i] * y[1, i] * (1. - y[2, i] / 3.)
            outfile.write("%le %le\n" % (V, (V * y[2, i]) / (3. - y[2, i])))


# ========================
# Main Loop
# ========================
def run_neqs7_models():
    summary_records = []

    for lam4 in lam4_set:
        print(f"\n=== Running λ4 = {lam4:.1e} ===")

        # Output directories
        OUTDIR = f"{BASE_OUTDIR}/lam4_{lam4:.1e}"
        os.makedirs(OUTDIR, exist_ok=True)
        OUTFILE1_NAME = f"{OUTDIR}/test_nr_neqs{NEQS}.dat"
        OUTFILE2_NAME = f"{OUTDIR}/test_esigma_neqs{NEQS}.dat"

        try:
            outfile1 = open(OUTFILE1_NAME, "w")
            outfile2 = open(OUTFILE2_NAME, "w")
        except IOError as e:
            print("Could not open output files: ", e)
            sys.exit()

        if SPECTRUM:
            u_s = np.empty((2, knos))
            u_t = np.empty((2, knos))
            y_final = np.empty(NEQS + 1)
            spec_count = 0

        calc = Calc()
        iters = 0
        points = 0
        outcount = 0
        asymcount = 0
        nontrivcount = 0
        insuffcount = 0
        noconvcount = 0
        badncount = 0
        errcount = 0
        savedone = 0

        while nontrivcount < NUMPOINTS:
            iters += 1
            if iters > 200:
                break

            if iters % 100 == 0:
                print(f"  Iter {iters}, nontriv={nontrivcount}")

            yinit, calc.Nefolds = pick_init_vals(lam4)
            y = yinit.copy()

            path = np.array([[]])
            N = np.array([])

            t0 = time.perf_counter()
            calc.ret = calcpath(calc.Nefolds, y, path, N, calc)
            t1 = time.perf_counter()
            print(f"calcpath runtime: {t1 - t0:.4f} s")
            print(f"  -> {calc.ret}")

            print("\n=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===")
            print(f"Initial values (yinit): {yinit}")
            print(f"  φ0 = {yinit[0]:.6e}")
            print(f"  H0 = {yinit[1]:.6e}")
            print(f"  ε0 = {yinit[2]:.6e}")
            print(f"  σ0 = {yinit[3]:.6e}")
            print(f"  λ₂ = {yinit[4]:.6e}")
            if len(yinit) > 5:
                print(f"  λ₃ = {yinit[5]:.6e}")
            if len(yinit) > 6:
                print(f"  λ₄ = {yinit[6]:.6e}")
            print(f"Initial Nefolds = {calc.Nefolds:.3f}")

            try:
                import pygsl.odeiv as odeiv
                s = odeiv.step_rk4(len(yinit), derivs1)
                c = odeiv.control_y_new(s, 1e-8, 1e-8)
                print(f"GSL integrator class: {s.__class__.__name__}")
                print(f"Expected tolerances: atol=1e-8, rtol=1e-8")
            except Exception as e:
                print("Could not check GSL integrator:", e)

            print("===============================================\n")

            if calc.ret == "asymptote":
                asymcount += 1
                if asymcount > 100:
                    print("Too many asymptotes, stopping")
                    break
                continue

            if calc.ret == "nontrivial":
                r = tsratio(y)
                ns = specindex(y)
                alpha_s = dspecindex(y)
                outfile1.write(f"{r:.10f} {ns:.10f} {alpha_s:.10f}\n")
                outfile1.flush()

                for i in range(NEQS):
                    outfile2.write("%le " % y[i])
                outfile2.write("%f\n" % calc.Nefolds)
                outfile2.flush()

                points += 1
                savedone = 0
                nontrivcount += 1

                if SPECTRUM and we_should_calc_spec(y):
                    print(f"  ns = {specindex(y):.3f}")
                    print(f"  -> Evaluating spectrum {spec_count}")

                    y_final[:NEQS] = path[:NEQS, 3]
                    y_final[NEQS] = N[3]

                    t0 = time.perf_counter()
                    spectrum_status = spectrum(
                        y_final, y, u_s, u_t, calc.Nefolds,
                        derivs1, scalarsys, tensorsys
                    )
                    t1 = time.perf_counter()
                    print(f"spectrum runtime: {t1 - t0:.4f} s")

                    if spectrum_status:
                        errcount += 1

                    spec_s_name = f"{OUTDIR}/spec_s{spec_count:03d}_neqs{NEQS}.dat"
                    spec_t_name = f"{OUTDIR}/spec_t{spec_count:03d}_neqs{NEQS}.dat"
                    np.savetxt(spec_s_name, u_s[:, :knos].T)
                    np.savetxt(spec_t_name, u_t[:, :knos].T)
                    spec_count += 1

                if SPECTRUM:
                    print(f"  -> Before normalization: y[1] = {y[1]:.6e}")
                    for j in range(calc.npoints):
                        path[0, j] = path[0, j] - path[0, calc.npoints - 1]
                        path[1, j] = path[1, j] * y[1]

                    print(f"  -> After normalization: max(path[1,:]) = {np.max(path[1,:]):.6e}")

                path_name = f"{OUTDIR}/path_neqs{NEQS}_lam4{lam4:.1e}_{outcount:03d}.dat"
                print(f"  -> Saving path {path_name}")
                save_path(path, N, calc.npoints, path_name)
                outcount += 1

                summary_records.append({
                    "lam4": lam4,
                    "r": r,
                    "n_s": ns,
                    "alpha_s": alpha_s,
                    "Nefolds": calc.Nefolds
                })

            elif calc.ret == "insuff":
                insuffcount += 1
            elif calc.ret == "noconverge":
                noconvcount += 1
            else:
                errcount += 1

        outfile1.close()
        outfile2.close()

    summary_df = pd.DataFrame(summary_records)
    summary_file = f"{BASE_OUTDIR}/neqs{NEQS}_summary.csv"
    summary_df.to_csv(summary_file, index=False)
    print(f"\nSummary written to {summary_file}")


print('env', sys.executable)

import platform, numpy, scipy, pandas, matplotlib
print(f"Python: {platform.python_version()}")
print(f"CPU Archachitecture: {platform.machine()}")
print(f"NumPy: {numpy.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"pandas: {pandas.__version__}")
print(f"matplotlib: {matplotlib.__version__}")

%time run_neqs7_models()